Pydantic model of the ARLEM specification of an AR education activity.
Full spec, including some complications and extra fields that may not be needed in simpler education scenarios

In [ ]:
from typing import List, Optional, Literal, Union, Set
from pydantic import BaseModel, Field, HttpUrl, field_validator, model_validator


In [ ]:

# ==========================================
# PART 1: WORKPLACE MODEL (The Environment)
# ==========================================

class POI(BaseModel):
    """
    Defines a specific point of interest on a physical object relative to its origin.
    """
    # Constraint: ID is Alphanumeric, Character String (100) 
    id: str = Field(..., max_length=100, description="Unique identifier for this specific point.")
    # Constraint: magnitude < 1000.00
    x_offset: float = Field(0.0, ge=-1000.00, le=1000.00, description="Offset on the x-axis in cm.")
    y_offset: float = Field(0.0, ge=-1000.00, le=1000.00, description="Offset on the y-axis in cm.")
    z_offset: float = Field(0.0, ge=-1000.00, le=1000.00, description="Offset on the z-axis in cm.")
    # Constraint: angles 0-360.0
    x_rotation: float = Field(0.0, ge=0.0, le=360.0, description="Pitch in Euler angles (degrees).")
    y_rotation: float = Field(0.0, ge=0.0, le=360.0, description="Yaw in Euler angles (degrees).")
    z_rotation: float = Field(0.0, ge=0.0, le=360.0, description="Roll in Euler angles (degrees).")

    @model_validator(mode='after')
    def validate_default_origin(self):
        """
        Enforce zero offset if Id is "default"
        """
        if self.id == "default":
            # Check for nonzero offsets
            if any([self.x_offset != 0, self.y_offset != 0, self.z_offset != 0]):
                raise ValueError(
                    "If POI id is 'default', it must be placed at the origin (0,0,0), rotation is allowed"
                )
        return self

class Detectable(BaseModel):
    """
    Configuration for the computer vision system to recognize objects.
    """
    id: str = Field(..., max_length=100, description="Unique identifier referenced by Things, Places, or Persons.")
    type: Literal["marker", "anchor"] = Field(..., description="tracking method, 'marker' is an image target, 'anchor' spatial anchor relative to workplace origin")
    sensor: Optional[Literal["tracking", "mapping"]] = Field("tracking", description="sensor subsystem to use, image tracking or spatial mapping).")
    # Not sure if HttpUrl will work with the LLM filling it in
    url: Optional[HttpUrl] = Field(None, max_length=2000, description="URL to an asset bundle defining the defining image marker or anchor data.")

class Tangible(BaseModel):
    """
    place or thing in the workspace
    """
    id: str = Field(..., max_length=100, description="Unique identifier for the tangible")
    name: str = Field(..., max_length=1000, description="human-readable short description of place or thing; or name of the person")
    detectable: str = Field(None, max_length=100, description="ID of the Detectable for this tangible")
    pois: Optional[List[POI]] = Field(default_factory=list, description="List of Points of Interest for this tangible")

class Person(Tangible):
    """
    A human user or instructor in the workplace.
    """
    twitter: Optional[str] = Field(None, max_length=15, description="Twitter handle for the person.")
    mbox: Optional[str] = Field(None, max_length=254, description="Email or ID used for xAPI logging identification.")
    persona: Optional[str] = Field(None, max_length=1000, description="Role or group membership (e.g., 'learner', 'instructor').")

class SensorData(BaseModel):
    """
    Definition of a data stream variable.
    """
    key: str = Field(..., max_length=100, description="The variable identifier/topic in the data stream.")
    type: Optional[Literal["string", "float", "integer", "boolean"]] = Field("string", description="The data type of the variable")

class Sensor(BaseModel):
    """
    IoT sensor configuration for receiving external data.
    """
    id: str = Field(..., max_length=100, description="Unique identifier for the sensor.")
    url: str = Field(...,
                     pattern=r"^[a-zA-Z0-9+.-]+:\/\/[a-zA-Z0-9.-]+(:[0-9]{1,5})?$",
                     max_length=1000,
                     description="Connection URL, of the form 'protocol://host.domain[:port]'",
                     examples=["mqtt://test.mosquitto.org:1883"])
    username: Optional[str] = Field(None, max_length=100, description="Auth username for the sensor stream.")
    password: Optional[str] = Field(None, max_length=100, description="Auth password for the sensor stream.")
    data: List[SensorData] = Field(default_factory=list, min_items=1, description="List of data variables to listen for.")

class Device(BaseModel):
    """
    Hardware available for the experience (e.g., HoloLens, Tablet).
    """
    id: str = Field(..., max_length=100, description="Unique identifier for the device.")
    type: str = Field(..., max_length=100, description="Type of device (e.g., 'hololens', 'tablet').")
    name: str = Field(..., max_length=1000, description="Human-readable name of the device.")
    owner: Optional[str] = Field(None, max_length=100, description="Person ID of the device owner.")
    url: Optional[str] = Field(...,
                     pattern=r"^[a-zA-Z0-9+.-]+:\/\/[a-zA-Z0-9.-]+(:[0-9]{1,5})?$",
                     max_length=1000,
                     description="Connection URL, of the form 'protocol://host.domain[:port]'",
                     examples=["mqtt://test.mosquitto.org:1883"])
    topic: Optional[str] = Field(None, max_length=1000, description="MQTT channel name this device listens to.")
    username: Optional[str] = Field(None, max_length=100, description="Auth username")
    password: Optional[str] = Field(None, max_length=100, description="Auth password")

class App(BaseModel):
    """
    External web apps or widgets.
    """
    id: str = Field(..., max_length=100, description="Unique identifier for the app")
    type: Literal["widget", "app", "prefab"] = Field(..., description="Type of application: html widget, launch command, or app prefab")
    name: str = Field(..., max_length=100, description="Human-readable description")
    url: str = Field(...,
                     pattern=r"^[a-zA-Z0-9+.-]+:\/\/[a-zA-Z0-9.-]+$",
                     max_length=1000,
                     description="URL of the manifest file, launch command, or download link of app prefab",
                     examples=["mqtt://test.mosquitto.org:1883"])

class Primitive(BaseModel):
    """
    Definition of what augmentations are supported in this workspace: audio, video, models, animations, or text labels.
    Note, this does not correspond with specific instances, like an image, but defines if any kind of image display is supported
    """
    id: Literal["animation", "image", "video", "audio", "label"] = Field(..., description="Unique media type of the augmentation.")
    # Optional size fields for bounding boxes
    x_size: Optional[float] = Field(None, gte=0.01, lte=1000.00, description="image or object size on X axis (cm)")
    y_size: Optional[float] = Field(None, gte=0.01, lte=1000.00, description="image or object size on Y axis (cm)")
    z_size: Optional[float] = Field(None, gte=0.01, lte=1000.00, description="image or object size on Z axis (cm)")
    volume: Optional[int] = Field(None, gte=0, lte=100, description="Volume for audio primatives")

class Predicate(BaseModel):
    """
    reusable instructional augmentations, like an image or animation, to be used in an activity
    """
    id: str = Field(..., max_length=100, description="Unique id for this instructional augmentation")
    type: Literal["animation", "image", "video", "audio", "label"] = Field(..., description="what type of primitive, instance of Primitive.id, cannot be a value not defined in the workplace primitives list")
    scale: float = Field(1.0, gte=0.01, lte=1000.00, description="normalized scale factor applied to primitive on all axes")
    # Don't know if max_length str validator works with HttpUrl class
    url: HttpUrl = Field(None, max_length=1000, description="Source URL for the file (GLB, fbx, MP4, PNG, etc.).")

class Warning(BaseModel):
    """
    ISO 7010 Hazard signs.
    """
    id: str = Field(..., max_length=100, description="Unique id for this instructional augmentation")
    type: Literal["animation", "image", "video", "audio", "label"] = Field(..., description="what type of primitive, instance of Primitive.id, cannot be a value not defined in the workplace primitives list")
    scale: float = Field(1.0, gte=0.01, lte=1000.00, description="normalized scale factor applied to primitive on all axes")
    symbol: str = Field(..., max_length=100, description="Name of the symbol prefab")

class Workplace(BaseModel):
    """
    The static environment definition containing all available resources.
    """
    id: str = Field(..., max_length=100, description="Unique identifier for this workplace")
    name: str = Field(..., max_length=1000, description="Descriptive name of the workplace")
    origin: str = Field(..., max_length=100, description="ID of the Detectable that serves as the World Origin")
    things: List[Tangible] = Field(default_factory=list, description="Physical tools, machines, or materials")
    places: List[Tangible] = Field(default_factory=list, description="Locations or zones in the workplace")
    persons: List[Person] = Field(default_factory=list, min_items=1, description="Users involved in the scenario")
    sensors: Optional[List[Sensor]] = Field(default_factory=list, description="Connected IoT devices")
    devices: List[Device] = Field(default_factory=list, min_items=1, description="AR hardware used for delivery")
    apps: Optional[List[App]] = Field(default_factory=list, description="External widgets or apps")
    detectables: List[Detectable] = Field(default_factory=list, min_items=1, description="Markers or Anchors used for tracking")
    primitives: List[Primitive] = Field(default_factory=list, min_items=1, description="Supported media types for predicates")
    predicates: List[Predicate] = Field(default_factory=list, min_items=1, description="A specific media primitive augmentation that corresponds to a verb or action")
    warnings: List[Warning] = Field(default_factory=list, description="Supported Warnings")

    @model_validator(mode='after')
    def validate_workplace(self):
        """
        Checks that the origin has a valid detectable,
        Places, things, persons reference valid detectables,
        and predicates reference valid primitive types
        """
        # Get valid ids
        valid_detectable_ids = {d.id for d in self.detectables}
        valid_primitive_ids = {p.id for p in self.primitives}

        # Origin validation
        if self.origin not in valid_detectable_ids:
            raise ValueError(f"Workplace origin '{self.origin}' is not defined in the 'detectables' list")
        
        # Tangible validation
        # Helper function to validate list of tangibles
        def validate_tangible_list(items: list, tangible_type: str):
            # no two tangibles of  the same type should have the same detectable
            used_detectables_in_kind: Set[str] = set()

            for item in items:
                # Check existence
                if item.detectable not in valid_detectable_ids:
                    raise ValueError(
                        f"The {tangible_type} '{item.id}' references a missing detectable '{item.detectable}'."
                    )
                
                # Check uniqueness within type
                if item.detectable in used_detectables_in_kind:
                    raise ValueError(
                            f"The {tangible_type} {item.id} is trying to use the same detectable '{item.detectable}' as another {tangible_type}. "
                            f"Detectable IDs must be unique within the {tangible_type} list."
                        )
        
        validate_tangible_list(self.things, "Thing")
        validate_tangible_list(self.places, "Place")
        validate_tangible_list(self.persons, "Person")

        # Predicate Validation
        for predicate in self.predicates:
            if predicate.type not in valid_primitive_ids:
                raise ValueError(
                    f"Predicate '{predicate.id}' has type '{predicate.type}', which is not a defined Primitive ID."
                )
        return self


# ==========================================
# PART 2: ACTIVITY MODEL (The Logic)
# ==========================================

class Instruction(BaseModel):
    """
    Text/Audio instructions displayed to the user.
    """
    title: str = Field(..., description="Short headline of the instruction.")
    description: str = Field(..., description="Detailed narrative text.")

class Activate(BaseModel):
    """
    Command to spawn an object or effect.
    """
    target: str = Field(..., description="ID of the Thing, Place, Person, or Action to apply this to.")
    type: Literal["primitive", "predicate", "warning", "action"] = Field(..., description="Category of the object being activated.")
    augmentation: Optional[str] = Field(None, description="ID of the specific augmentation (from Workplace) to display.")
    poi: Optional[str] = Field(None, description="ID of the POI on the target where the augmentation appears.")
    url: Optional[HttpUrl] = Field(None, description="Override URL for dynamic content loading.")
    text: Optional[str] = Field(None, description="Content for text label primitives.")
    state: Optional[str] = Field(None, description="Keyframe or state ID for animations.")
    viewport: Optional[Literal["actions", "reactions", "warnings"]] = Field(None, description="If set, attaches object to the screen/HUD instead of the world.")
    option: Optional[str] = Field(None, description="Configuration option (e.g. direction 'up' for a pointer).")

class Deactivate(BaseModel):
    """
    Command to remove an object or effect.
    """
    target: str = Field(..., description="ID of the target to clear, or '*' for all.")
    type: Optional[Literal["primitive", "predicate", "warning", "action"]] = Field(None, description="Filter by type to remove.")
    augmentation: Optional[str] = Field(None, description="Specific augmentation ID to remove.")

class Message(BaseModel):
    """
    Communication between devices or users.
    """
    target: str = Field(..., description="ID of the Person or Device receiving the message.")
    type: Literal["person", "device", "sensor"] = Field(..., description="Type of recipient.")
    text: Optional[str] = Field(None, description="Body of the message.")
    key: Optional[str] = Field(None, description="Topic key for sensor/device broadcasting.")
    launch: Optional[str] = Field(None, description="ID of an action to trigger on the recipient device.")

class IfLogic(BaseModel):
    """
    Conditional logic based on xAPI statements (analytics).
    """
    url: str = Field(..., description="Query URL to check for xAPI statements.")
    then_action: str = Field(..., alias="then", description="Action ID to trigger if condition is met.")
    else_action: str = Field(..., alias="else", description="Action ID to trigger if condition is NOT met.")
    min_results: Optional[int] = Field(None, alias="min", description="Minimum number of statements required.")
    max_results: Optional[int] = Field(None, alias="max", description="Maximum number of statements allowed.")

class Trigger(BaseModel):
    """
    Event listener that causes the transition to the next step.
    """
    mode: Literal["click", "voice", "detect", "sensor"] = Field(..., description="Input method (UI click, voice command, object detection, or IoT data).")
    id: str = Field(..., description="ID of the entity to listen to (Action ID, Tangible ID, or Sensor ID).")
    duration: Optional[int] = Field(None, description="Time in ms required for gaze/detection to trigger.")
    key: Optional[str] = Field(None, description="Variable key to check (for sensor mode).")
    value: Optional[str] = Field(None, description="Threshold value to check (for sensor mode).")
    operator: Optional[Literal["equal", "exceed", "below", "between"]] = Field("equal", description="Comparison operator (for sensor mode).")

class ActionFlow(BaseModel):
    """
    The set of commands executed when entering or exiting a step.
    """
    remove_self: bool = Field(False, alias="removeSelf", description="If True, removes the current instruction/UI immediately.")
    activate: List[Activate] = Field(default_factory=list, description="List of items to display.")
    deactivate: List[Deactivate] = Field(default_factory=list, description="List of items to hide.")
    messages: List[Message] = Field(default_factory=list, description="Messages to send.")
    if_logic: List[IfLogic] = Field(default_factory=list, alias="if", description="Conditional logic checks.")

class Action(BaseModel):
    """
    A single step in the activity workflow.
    """
    id: str = Field(..., description="Unique ID for this step.")
    instruction: Optional[Instruction] = Field(None, description="Text displayed to the user during this step.")
    enter: Optional[ActionFlow] = Field(None, description="Commands executed immediately when this step starts.")
    exit: Optional[ActionFlow] = Field(None, description="Commands executed when the Trigger conditions are met.")
    triggers: List[Trigger] = Field(default_factory=list, description="Conditions that cause the step to complete.")

class Activity(BaseModel):
    """
    The root object defining the logic flow of the AR experience.
    """
    id: str = Field(..., description="Unique ID for the activity.")
    name: str = Field(..., description="Human-readable title.")
    description: Optional[str] = Field(None, description="Overview of the activity.")
    language: str = Field("en", min_length=2, max_length=2, description="ISO 639-1 language code.")
    workplace: Union[HttpUrl, str] = Field(..., description="URL or Reference to the Workplace definition file.")
    start: str = Field(..., description="ID of the first Action to execute.")
    actions: List[Action] = Field(default_factory=list, description="All available steps/states in this activity.")